# Mini-Challenge DEL: Hyperparameter Tuning und Model Evaluation

In [ ]:
# System & Basics
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# Deep Learning Framework (PyTorch)
import torch
from torch.utils.data import DataLoader

# Computer Vision & Datasets
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# Visualisierung & Tracking
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Some EDA

In [ ]:
# Laden des svhn
def load_svhn_data(batch_size=64):
    transform = transforms.Compose([
        transforms.ToTensor(),        
    ])

    train_dataset = datasets.SVHN(root='../data', split='train', download=True, transform=transform)
    test_dataset = datasets.SVHN(root='../data', split='test', download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    return train_loader, test_loader

train_loader, test_loader = load_svhn_data()


In [ ]:
# EDA: Visualize some samples from the dataset
data_iter = iter(train_loader)
num_samples_to_show = 5

images, labels = next(data_iter)

plt.figure(figsize=(10, 2))
for i in range(num_samples_to_show):
    plt.subplot(1, num_samples_to_show, i + 1)
    plt.imshow(np.transpose(images[i].numpy(), (1, 2, 0)) * 0.5 + 0.5)
    plt.title(f'Label: {labels[i].item()}')
    plt.axis('off')
plt.show()


In [ ]:
# EDA: Klassen-Balanciertheit + Featurevergleich (RGB-Intensitäten je Klasse)

def prepare_labels(dataset):
    labels = np.array(dataset.labels)
    labels = np.where(labels == 10, 0, labels)
    return labels

def class_dist(labels, split_name="Train"):
    # Klassenverteilung
    class_counts = pd.Series(labels).value_counts().sort_index()
    class_pct = (class_counts / class_counts.sum() * 100).round(2)

    dist_df = pd.DataFrame({
        "Klasse": class_counts.index,
        "Anzahl": class_counts.values,
        "Anteil_%": class_pct.values
    })

    return dist_df, class_counts



def feature_comp(ds, labels, split_name="Train"):
    images = ds.data.astype(np.float32) / 255.0  # (N, C, H, W)

    # Featurevergleich: mittlere RGB-Intensitäten je Bild und Klasse
    mean_r = images[:, 0].mean(axis=(1, 2))
    mean_g = images[:, 1].mean(axis=(1, 2))
    mean_b = images[:, 2].mean(axis=(1, 2))
    mean_intensity = images.mean(axis=(1, 2, 3))

    feat_df = pd.DataFrame({
        "Klasse": labels,
        "Mean_R": mean_r,
        "Mean_G": mean_g,
        "Mean_B": mean_b,
        "Mean_Intensity": mean_intensity
    })

    class_feat = feat_df.groupby("Klasse")[["Mean_R", "Mean_G", "Mean_B", "Mean_Intensity"]].mean()
    print(f"\n=== {split_name}: Durchschnittliche Feature-Werte je Klasse ===")
    print(class_feat.round(4))

    # Visualisierungen
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.barplot(data=dist_df, x="Klasse", y="Anzahl", ax=axes[0], palette="viridis", hue="Anteil_%", dodge=False)
    axes[0].set_title(f"{split_name}: Klassenverteilung")
    axes[0].set_xlabel("Klasse")
    axes[0].set_ylabel("Anzahl")

    class_feat_plot = class_feat.reset_index().melt(
        id_vars="Klasse",
        value_vars=["Mean_R", "Mean_G", "Mean_B"],
        var_name="Feature",
        value_name="Mittelwert"
    )
    sns.barplot(data=class_feat_plot, x="Klasse", y="Mittelwert", hue="Feature", ax=axes[1], palette="Set2",)
    axes[1].set_title(f"{split_name}: RGB-Featurevergleich je Klasse")
    axes[1].set_xlabel("Klasse")
    axes[1].set_ylabel("Mittlere Intensität")
    axes[1].legend(title="Feature")

    plt.tight_layout()
    plt.show()


# EDA für Train und Test
train_labels = prepare_labels(train_loader.dataset)
test_labels = prepare_labels(test_loader.dataset)

for split_name, loader in [("Train", train_loader), ("Test", test_loader)]:
    dist_df, class_counts = class_dist(labels, split_name=split_name)
    print(f"\n=== {split_name}: Klassenverteilung ===")
    print(dist_df.to_string(index=False))
    imbalance_ratio = class_counts.max() / class_counts.min()
    print(f"Imbalance-Ratio (max/min): {imbalance_ratio:.2f}")

feature_comp(train_loader.dataset, train_labels, split_name="Train")
feature_comp(test_loader.dataset, test_labels, split_name="Test")


In [ ]:
# Preprocessing der Daten
# calculate mean and std for normalization for each channel
def get_stats(loader):
    sum_, squared_sum, batches = 0, 0, 0
    
    for images, _ in loader:
        # images: [batch_size, 3, 32, 32]
        sum_ += torch.mean(images, dim=[0, 2, 3])
        squared_sum += torch.mean(images**2, dim=[0, 2, 3])
        batches += 1
        
    mean = sum_ / batches
    std = (squared_sum / batches - mean**2)**0.5
    return mean, std

# Berechnung
mean, std = get_stats(train_loader)
print(f"Mean: {mean}")
print(f"Std: {std}")


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

train_set = datasets.SVHN(root='../data', split='train', download=True, transform=transform)
test_set = datasets.SVHN(root='../data', split='test', download=True, transform=transform)
    
train_loader = DataLoader(train_set, batch_size=4, shuffle=True)
test_loader = DataLoader(test_set, batch_size=4, shuffle=False)

# 2. Visualisieren
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Normalisiertes Bild Anzeigen:
plt.imshow(images[0].permute(1, 2, 0)) # Permute für (H, W, C)
plt.title(f"Label: {labels[0]}")
plt.show()

## Metriken

Da der vorliegende Datensatz eine starke Klassenimbalance aufweist, ist die Accuracy als alleinige Metrik nicht aussagekräftig. Um die Modellleistung über alle Kategorien hinweg objektiv zu bewerten, fokussiere ich mich stattdessen auf den F1-Score sowie Precision und Recall.
Da es sich um ein Multiclass-Problem handelt, wird der CrossEntropyLoss als Verlustfunktion verwendet. Um sicherzustellen, dass auch minderbesetzte Klassen angemessen berücksichtigt werden, kommen bei der Evaluation die Macro-Averages (Macro F1, Macro Precision und Macro Recall) zum Einsatz. 


## 3. Basisfunktionalität

Die Basisfunktionalität wurde wie folgt umgesetzt:

1. **Konfiguration über `RunConfig`**
    - Zentrale Parameter (z. B. `learning_rate`, `batch_size`, `epochs`, `device`) werden in `baseline_config` gebündelt.

2. **Baseline-Training über `run_baseline(...)`**

3. **Experiment-Tracking mit Weights & Biases**
    - Aktiviert über `use_wandb=True` in der Config.
    - Projektname: `DEL`, Run-Name: `baseline-example`.

4. **Auswertung über `WandbRuns`**
    - lädt Runs aus W&B:
    ```python
    wandb_runs = WandbRuns(project="DEL")
    overview_df = wandb_runs.runs_overview(top_k=10)
    ```
    - Zusätzlich vergleichst du Metriken (z. B. `val/loss`) direkt zwischen Runs.

---

## Struktur

```text
src/
├── training.py
│   ├── RunConfig            # Konfigurationsdataclass für Training
│   └── run_baseline(...)    # Führt Baseline-Training + Evaluation aus
├── wandb_tools.py
│   └── WandbRuns            # Laden, Filtern, Vergleichen von W&B-Runs
├── model.py            # Modellarchitekturen
└── metrics.py           # F1, Precision, Recall (Macro)
```

In [ ]:
# Basisfunktionalitaet: Das Notebook dokumentiert und visualisiert,
from src.training import run_baseline, RunConfig
from src.wandb_tools import WandbRuns

baseline_config = RunConfig(
    learning_rate=1e-2,
    batch_size=64,
    epochs=5,
    use_wandb=True,
    wandb_project="DEL",
    run_name="baseline-example",
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_seed=42,
    tags=["baseline"],
)

baseline_result = run_baseline(
    config=baseline_config, 
    train_loader=train_loader, 
    )

In [ ]:
# Overfitting-Test: Kann das Basismodell einen einzelnen Batch auswendig lernen?
from torch.utils.data import DataLoader, TensorDataset
from src.training import FitConfig

batch_images, batch_labels = next(iter(train_loader))
overfit_dataset = TensorDataset(batch_images.clone(), batch_labels.clone())
overfit_loader = DataLoader(overfit_dataset, batch_size=len(overfit_dataset), shuffle=False)

overfit_config = RunConfig(
    learning_rate=5e-2,
    batch_size=len(overfit_dataset),
    epochs=100,
    use_wandb=False,
    wandb_project="DEL",
    run_name="overfit-single-batch",
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_seed=42,
    tags=["overfit-check"],
)

overfit_result = run_baseline(
    config=overfit_config,
    fit_config=FitConfig(epochs=100),
    train_loader=overfit_loader,
    val_loader=overfit_loader,
)

history = overfit_result["history"]
print(f"Final train accuracy: {history['train_accuracy'][-1]:.4f}")
print(f"Final val accuracy: {history['val_accuracy'][-1]:.4f}")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")
